# 33 — Local, Server and Edge Deployment for Network LLMs

**Network LLM Engineer Certification — Deployment**

### Learning goals
- Compare local desktop, embedded, single-node server and scaled API deployment
- Understand Ollama/LM Studio/GPT4All/llama.cpp/vLLM-style roles
- Choose a deployment architecture from sovereignty, latency, throughput and hardware constraints

## Deployment is not one thing

A useful taxonomy:

1. **Local interactive** — engineer laptop/workstation.
2. **Demo/app** — Streamlit/Gradio/FastAPI-style UI/API.
3. **Server inference** — shared GPU endpoint.
4. **Edge** — constrained device close to data source.
5. **Managed model API** — external service.

Network AI often needs more than one.

## Local runtime landscape

### Ollama
Convenient local model management and API-style use; strong for developer workflows.

### LM Studio
Desktop application oriented toward model discovery, interactive testing and local API use.

### GPT4All
Desktop/local experimentation with emphasis on easy local use.

### llama.cpp
Low-level, highly portable C/C++ inference ecosystem strongly associated with GGUF; useful for CPU and heterogeneous local hardware.

### Transformers
Python model execution and experimentation; ideal for model research and training integration, not necessarily the best high-throughput server.

### vLLM-style server
GPU-oriented shared inference server emphasizing throughput, batching and OpenAI-compatible APIs.

The exact feature set evolves; benchmark the current release before standardizing.

## Networking deployment matrix

| Environment | Typical choice |
|---|---|
| Engineer MacBook | llama.cpp/Ollama/LM Studio + quantized model |
| Branch/local appliance | small quantized SLM, tightly scoped tools |
| On-prem NOC GPU | vLLM-like serving + model router |
| Central enterprise | multi-GPU server + HA + observability |
| Sensitive read-only assistant | local/on-prem with local RAG |
| Burst/high capability | hybrid local + governed external API |

## OpenAI-compatible API as an abstraction

Many local/server runtimes expose an API shaped similarly to `/v1/chat/completions`.

That can decouple the AI NOC application from one specific model runtime, but compatibility is not perfect:
- tool calling,
- structured output,
- multimodal messages,
- reasoning fields,
- model names,
may differ.

In [ ]:
# Example deployment manifest — not tied to one runtime.
deployment = {
    "service":"network-llm",
    "location":"on-prem",
    "api_style":"OpenAI-compatible",
    "models":{
        "triage":"3B quantized",
        "reasoning":"8B/14B BF16 or quantized after benchmark",
    },
    "data_egress":"blocked by default",
    "tools":"read-only by default",
    "ha":"2 replicas",
}
import json
print(json.dumps(deployment, indent=2))

## Edge deployment

Edge makes sense when:
- connectivity to central inference is poor,
- privacy/locality is required,
- response latency must be very low,
- the task is narrow enough for a small model.

Do not put a giant reasoning agent on an edge device just because it is technically possible.
A small classifier or deterministic parser may be better.

### Exercise — Deployment architecture

Design the deployment for:
- 300 network engineers,
- two EU data centers,
- no sensitive config data may leave company infrastructure,
- interactive assistant target p95 < 4 s,
- 100 concurrent requests at peak,
- local Mac fallback when disconnected.

Select runtimes/model tiers and identify what must be benchmarked.